In [1]:
%cd ~/
!rm -rf SMILES-2026-Hallucination-Detection
!git clone https://github.com/p4sttt/SMILES-2026-Hallucination-Detection.git
%cd SMILES-2026-Hallucination-Detection
!ls -ls

/root
Cloning into 'SMILES-2026-Hallucination-Detection'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 50 (delta 21), reused 23 (delta 14), pack-reused 12 (from 1)
Receiving objects: 100% (50/50), 460.33 KiB | 11.23 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/root/SMILES-2026-Hallucination-Detection
total 104
 8 -rw-r--r-- 1 root root  5562 May  8 11:39 aggregation.py
 4 drwxr-xr-x 2 root root  4096 May  8 11:39 data
16 -rw-r--r-- 1 root root 12572 May  8 11:39 evaluate.py
 4 -rw-r--r-- 1 root root  1063 May  8 11:39 LICENSE
 4 -rw-r--r-- 1 root root   113 May  8 11:39 main.py
 4 -rw-r--r-- 1 root root  1381 May  8 11:39 model.py
 8 -rw-r--r-- 1 root root  4883 May  8 11:39 probe.py
 4 -rw-r--r-- 1 root root   186 May  8 11:39 pyproject.toml
 8 -rw-r--r-- 1 root root  5713 May  8 11:39 README.md
 4 -rw-r--r-- 1 root root   112 May  8 11:39 requirements.txt
24 -rw-r--r-- 1 ro

# SMILES-2026 Hallucination Detection

Colab-friendly notebook version of `solution.py`. Run the cells from top to bottom from the repository root. The notebook writes the same `results.json` and `predictions.csv` files as the script.

## 1. Setup

In Google Colab, use a GPU runtime: `Runtime -> Change runtime type -> T4 GPU` or another available GPU. If you cloned the repository manually, `cd` into the repository root before running the rest of the notebook.

In [2]:
from pathlib import Path

if not Path("requirements.txt").exists():
    raise FileNotFoundError(
        "Run this notebook from the repository root, where requirements.txt is located."
    )

!pip install -q -r requirements.txt

## 2. Imports And Config

In [3]:
import time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from aggregation import aggregation_and_feature_extraction
from evaluate import print_summary, run_evaluation, save_predictions, save_results
from model import MAX_LENGTH, get_model_and_tokenizer
from probe import HallucinationProbe
from splitting import split_data

DATA_FILE = "./data/dataset.csv"
OUTPUT_FILE = "results.json"
BATCH_SIZE = 4
USE_GEOMETRIC = False
TEST_FILE = "./data/test.csv"
PREDICTIONS_FILE = "predictions.csv"

assert OUTPUT_FILE == "results.json"
assert PREDICTIONS_FILE == "predictions.csv"

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device        : {device}")
print(f"Data          : {DATA_FILE}")
print(f"Max length    : {MAX_LENGTH} tokens")
print(f"Geometric feats: {USE_GEOMETRIC}")

Device        : cuda
Data          : ./data/dataset.csv
Max length    : 512 tokens
Geometric feats: False


## 3. Load Dataset

In [5]:
df = pd.read_csv(DATA_FILE)

all_texts = [f"{row['prompt']}{row['response']}" for _, row in df.iterrows()]
all_labels = np.array([int(float(h)) for h in df["label"]])

n_total = len(all_labels)
print(
    f"Loaded {n_total} samples "
    f"({all_labels.sum()} hallucinated / {(all_labels == 0).sum()} truthful)"
)
print(f"Columns : {df.columns.tolist()}")
print(f"Rows    : {len(df)}")
print(f"Labels  : {dict(df['label'].value_counts().sort_index())}")

row0 = df.iloc[0]
print("\nprompt preview:")
print(row0["prompt"][:500])
print("\nresponse preview:")
print(row0["response"][:300])
label_str = "hallucinated" if int(row0["label"]) else "truthful"
print(f"\nlabel: {int(row0['label'])} ({label_str})")

Loaded 689 samples (483 hallucinated / 206 truthful)
Columns : ['prompt', 'response', 'label']
Rows    : 689
Labels  : {0.0: np.int64(206), 1.0: np.int64(483)}

prompt preview:
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Given the context, answer the question in a single brief but complete sentence.

This is the most common method of construction procurement and is well established and recognized. In this arrangement, the architect or engineer acts as the project coordinator. His or her role is to design the works, prepare the specifications and produce construction drawings, administer the contract, tender the works, and manage the works

response preview:
An architect or engineer has a direct relationship with the subcontractor.<|endoftext|>

label: 1 (hallucinated)


## 4. Load Model And Define Feature Extraction

In [6]:
model, tokenizer = get_model_and_tokenizer()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.to(device)
model.eval()

[Model] Loading 'Qwen/Qwen2.5-0.5B' ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [7]:
def extract_features(texts: list[str], desc: str) -> tuple[np.ndarray, float]:
    features = []
    t0 = time.time()

    for start in tqdm(range(0, len(texts), BATCH_SIZE), desc=desc, unit="batch"):
        batch_texts = texts[start : start + BATCH_SIZE]
        encoding = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        hidden = torch.stack(outputs.hidden_states, dim=1).float()
        mask = attention_mask.cpu()

        for i in range(hidden.size(0)):
            feat = aggregation_and_feature_extraction(
                hidden[i],
                mask[i],
                use_geometric=USE_GEOMETRIC,
            )
            features.append(feat.cpu())

        del input_ids, attention_mask, outputs, hidden, mask
        if device.type == "cuda":
            torch.cuda.empty_cache()

    elapsed = time.time() - t0
    X = np.vstack([f.numpy() for f in features])
    return X, elapsed

## 5. Extract Train Features

In [8]:
X, extract_time = extract_features(all_texts, desc="Extracting and aggregating")
y = all_labels

print(f"Done in {extract_time:.1f} s - {len(X)} feature vectors extracted")
print(f"Feature matrix : {X.shape} (feature_dim = {X.shape[1]})")
print(f"Geometric feats: {USE_GEOMETRIC}")

Extracting and aggregating:   0%|          | 0/173 [00:00<?, ?batch/s]

Done in 165.7 s - 689 feature vectors extracted
Feature matrix : (689, 24) (feature_dim = 24)
Geometric feats: False


## 6. Evaluate Probe

In [9]:
splits = split_data(y, df)

print(f"Splits : {len(splits)} fold(s)")
for i, (tr, va, te) in enumerate(splits):
    print(
        f"  Fold {i + 1}: train={len(tr)} "
        f"val={len(va) if va is not None else 'N/A'} test={len(te)}"
    )

fold_results = run_evaluation(splits, X, y, HallucinationProbe)

print_summary(fold_results, X.shape[1], len(X), extract_time)
save_results(fold_results, X.shape[1], len(X), extract_time, OUTPUT_FILE)

Splits : 1 fold(s)
  Fold 1: train=481 val=104 test=104

──────────────────────────────────────────────────
  Fold 1/1  —  train=481  val=104  test=104
──────────────────────────────────────────────────
  Baseline  — Acc: 70.19%  F1: 82.49%
  Probe train — Acc: 72.35%  F1: 82.57%  AUROC: 73.42%
  Probe val  — Acc: 72.12%  F1: 83.04%  AUROC: 68.05%
  Probe test — Acc: 71.15%  F1: 81.93%  AUROC: 65.31%

 Hallucination Detection — Evaluation Summary
  Checkpoint                           Accuracy      F1   AUROC
------------------------------------------------------------
  1. Majority-class baseline             70.19%  82.49%     N/A
  2. Probe (train split)                 72.35%  82.57%  73.42%
  3. Probe (val split)                   72.12%  83.04%  68.05%
  4. Probe (test split)                  71.15%  81.93%  65.31%
------------------------------------------------------------
  Feature dim  : 24
  Total samples: 689
  Folds        : 1
  Extract time : 165.7 s

★  Primary metric — T

## 7. Predict Competition Test Set

In [10]:
df_test = pd.read_csv(TEST_FILE)
test_texts = [f"{row['prompt']}{row['response']}" for _, row in df_test.iterrows()]
test_ids = df_test.index
print(f"Test set loaded: {len(test_texts)} samples")

X_test, test_extract_time = extract_features(
    test_texts,
    desc="Test extraction and aggregation",
)
print(f"Test features: {X_test.shape}; extracted in {test_extract_time:.1f} s")

Test set loaded: 100 samples


Test extraction and aggregation:   0%|          | 0/25 [00:00<?, ?batch/s]

Test features: (100, 24); extracted in 24.8 s


In [11]:
idx_non_test = np.unique(
    np.concatenate(
        [
            np.concatenate([idx_tr, idx_va]) if idx_va is not None else idx_tr
            for idx_tr, idx_va, _ in splits
        ]
    )
)

final_probe = HallucinationProbe()
final_probe.fit(X[idx_non_test], y[idx_non_test])
save_predictions(final_probe, X_test, test_ids, PREDICTIONS_FILE)

print(f"Saved {OUTPUT_FILE} and {PREDICTIONS_FILE}")

Predictions saved to 'predictions.csv'  (100 samples)
Saved results.json and predictions.csv
